# Spezialist für frühen Mais und Soja

**Autor:** jd · **Datum:** 2026-10-29 · **Issue:** –

**Hypothese:** Früher Mais und früher Soja sind das am häufigsten verwechselte Paar (Kultur früh 18 % falsch). Ein zweites Modell, das nur auf diesen vier Klassen (Mais/Soja × Emerge_VEarly/Early_Mid) trainiert ist, entscheidet die Zeilen neu, die die Referenz dorthin sortiert (`PairSpecialistClassifier`). Es kann seine ganze Kapazität auf die kleinen Unterschiede innerhalb der Gruppe verwenden.

**Ablauf:** Entschieden wird nur per CV gegen die Referenz (`combined_svm_aez_month`, gleiches Raster); validiert wird nur, wenn der Gewinn `MIN_CV_GAIN` (0.005) erreicht.

In [ ]:
%load_ext autoreload
%autoreload 2

import pandas as pd
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.svm import SVC

from awp2.config import (
    CROP_COL,
    EARLY_STAGES,
    EXTRA_TREES_N_ESTIMATORS,
    MIN_CV_GAIN,
    SEED,
    STAGE_COL,
)
from awp2.data import load_split
from awp2.evaluation import compare_predictions, evaluate, recall_by_class
from awp2.experiment import RunConfig, TuneConfig, out_of_fold_predictions, run, tune
from awp2.models.combined import make_combined_svm
from awp2.models.specialist import PairSpecialistClassifier, specialist_upper_bound
from awp2.plots import plot_cv_scores, plot_metric_comparison, plot_recall_comparison
from awp2.preprocessing import PreprocessingConfig

# A · Läufe

In [ ]:
SLUG = "combined_svm_specialist"
TOPIC = "Mais/Soja-Spezialist"
# reference: combined_svm_aez_month (d93cea89)
REFERENCE = PreprocessingConfig(use_spectral_standard_scale=True, use_aez_month=True)
GRID = {"estimator__C": [100.0, 300.0, 1000.0], "estimator__gamma": [0.001, 0.003, 0.01]}
y_train = load_split().y_train
reference = tune(
    make_combined_svm(),
    TuneConfig(
        name=f"{SLUG}_reference_search",
        approach="combined",
        description="Referenz-SVM, gleiches Raster.",
        param_grid=GRID,
        preprocessing=REFERENCE,
    ),
)
reference_cv = reference.best_score
print(f"Referenz CV: {reference_cv:.4f}")

### Obergrenze: Was brächte ein perfekter Spezialist?

Auf den Out-of-fold-Vorhersagen der Referenz: Jede Zeile, die in die Gruppe sortiert wird *und* wirklich dazugehört, wird richtig gesetzt. Fehler, die aus der Gruppe heraus- oder in sie hineinführen, bleiben.

In [ ]:
oof_reference = out_of_fold_predictions(reference.best_model, reference.config)
upper = specialist_upper_bound(y_train, oof_reference)
pd.DataFrame(
    {
        "Referenz (out-of-fold)": evaluate(y_train, oof_reference).model_dump(),
        "mit perfektem Spezialisten": evaluate(y_train, upper).model_dump(),
    }
)

### Spezialist: RBF-SVM oder Extra Trees

Die Referenz-SVM bleibt als Basis fest; gesucht werden nur die Einstellungen des Spezialisten.

In [ ]:
base = reference.best_model
candidates = {
    "Spezialist SVM": tune(
        PairSpecialistClassifier(base, SVC(class_weight="balanced")),
        TuneConfig(
            name=f"{SLUG}_svm_search",
            approach="combined",
            description="Referenz-SVM + SVM-Spezialist für frühen Mais/Soja.",
            param_grid={
                "specialist__C": [10.0, 100.0, 1000.0],
                "specialist__gamma": [0.001, 0.003, 0.01, 0.03],
            },
            preprocessing=REFERENCE,
        ),
    ),
    "Spezialist Extra Trees": tune(
        PairSpecialistClassifier(
            base,
            ExtraTreesClassifier(
                n_estimators=EXTRA_TREES_N_ESTIMATORS,
                class_weight="balanced",
                random_state=SEED,
                n_jobs=-1,
            ),
        ),
        TuneConfig(
            name=f"{SLUG}_et_search",
            approach="combined",
            description="Referenz-SVM + Extra-Trees-Spezialist für frühen Mais/Soja.",
            param_grid={"specialist__max_features": [0.1, 0.3, 0.6]},
            preprocessing=REFERENCE,
        ),
    ),
}

In [ ]:
searches = {"Referenz": reference, **candidates}
pd.DataFrame(
    {label: {"CV": result.best_score, **result.best_params} for label, result in searches.items()}
).T

# B · Erkenntnisse

## 1. CV-Score je Variante

In [ ]:
ax = plot_cv_scores(
    {label: result.candidates for label, result in searches.items()}, reference=reference_cv
)
ax.set_title(f"{TOPIC}: bester CV-Score je Variante (gestrichelt: Referenz)")

## 2. Gepaart auf dem Trainingsteil (out-of-fold)

In [ ]:
best_label = max(candidates, key=lambda label: candidates[label].best_score)
best = candidates[best_label]
oof = {
    "Referenz": out_of_fold_predictions(reference.best_model, reference.config),
    best_label: out_of_fold_predictions(best.best_model, best.config),
}
print(f"Beste Variante: {best_label}")
compare_predictions(y_train, oof["Referenz"], oof[best_label])

## 3. Zielfehler: früher Mais ↔ Soja

In [ ]:
early = y_train[STAGE_COL].isin(EARLY_STAGES)


def target_errors(pred: pd.DataFrame) -> pd.Series:
    swapped = (
        early
        & y_train[CROP_COL].isin(["corn", "soybean"])
        & pred[CROP_COL].isin(["corn", "soybean"])
        & (y_train[CROP_COL] != pred[CROP_COL])
    )
    return pd.Series(
        {
            "früher Mais ↔ Soja": swapped.sum(),
            "frühe Zeilen, Kultur falsch": (early & (y_train[CROP_COL] != pred[CROP_COL])).sum(),
            "alle Zeilen, Paar falsch": ((y_train != pred).any(axis=1)).sum(),
        }
    )


pd.DataFrame({label: target_errors(pred) for label, pred in oof.items()})

## 4. Recall je Klasse: wo ändert sich etwas?

In [ ]:
plot_recall_comparison(
    {label: recall_by_class(y_train, pred) for label, pred in oof.items()}, top=10
)

## 5. Entscheidung

In [ ]:
gain = best.best_score - reference_cv
print(f"CV-Gewinn {gain:+.4f} (Schwelle {MIN_CV_GAIN})")
if gain >= MIN_CV_GAIN:
    validated = run(
        best.best_model,
        RunConfig(
            name=SLUG,
            approach="combined",
            description=f"{TOPIC}: {best_label}, einmal auf der Validierung.",
            preprocessing=REFERENCE,
            tuning_run=best.run_id,
        ),
    )
    reference_val = run(
        reference.best_model,
        RunConfig(name="reference", approach="combined", preprocessing=REFERENCE, track=False),
    )
    display(
        plot_metric_comparison({"Referenz": reference_val.metrics, best_label: validated.metrics})
    )
else:
    print("Kein Gewinn über der Rauschschwelle – keine Validierung, Referenz bleibt.")

# Fazit

| | CV | früher Mais ↔ Soja (out-of-fold) |
| --- | --- | --- |
| Referenz-SVM | 0.864 | 64 |
| + SVM-Spezialist (`C=100`, `gamma=0.003`) | 0.865 | 64 |
| + Extra-Trees-Spezialist | 0.855 | – |

- **Obergrenze spürbar:** Ein perfekter Spezialist hebt die BAcc kombiniert auf dem Trainingsteil
  von 0.863 auf 0.887 (+0.024) – die Idee ist nicht von vornherein wertlos.
- **Erreicht wird nichts:** Der beste Spezialist ändert kaum Vorhersagen (14 : 12 Zeilen, p = 0.85)
  und korrigiert keine einzige zusätzliche frühe Mais/Soja-Zeile.
- Auch ein auf die vier Klassen spezialisiertes Modell findet die Trennung nicht in den Spektren.
  Zusammen mit den Boden-Indizes (Notebook 16) spricht das dafür, dass die Information in diesem
  Stadium fehlt: Der Bestand ist zu lückig, beide Kulturen sehen wie Boden aus.
- Keine Validierung; die Referenz bleibt.